# 61 — 时序模型：PatchTST 与 iTransformer

> 把 Transformer 用在时间序列上——PatchTST 和 iTransformer 是 2024 年的突破。

## 本章你将掌握

1. **时序问题**：预测、分类、异常检测
2. **PatchTST**：patch + Transformer
3. **iTransformer**：反转维度
4. **vs 传统方法**：ARIMA/Prophet/LSTM

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import torch
import torch.nn as nn
import torch.nn.functional as F
import warnings
warnings.filterwarnings('ignore')
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
np.random.seed(42)
torch.manual_seed(42)
print("环境准备完成 ✅")

## 1. 时间序列问题

### 1.1 任务类型

```
1. 预测 (Forecasting)
   给定历史 → 预测未来
   x[1:T] → x[T+1:T+H]

2. 分类 (Classification)
   给定序列 → 分类标签
   x[1:T] → label

3. 异常检测 (Anomaly Detection)
   给定序列 → 找异常点
   x[1:T] → {anomaly points}

4. 插补 (Imputation)
   给定有缺失的序列 → 填充缺失
   x[1:T] (有缺失) → x[1:T] (完整)
```

### 1.2 传统方法

```
ARIMA: 自回归+差分+移动平均
  - 适合平稳序列
  - 难处理多变量

Prophet: Facebook 的分解方法
  - 趋势+季节+假日
  - 适合商业数据

LSTM/GRU: RNN 变体
  - 能处理长序列
  - 但串行计算慢

Transformer: 自注意力
  - 并行计算
  - 长距离依赖
  - 但直接用效果不一定好
```

> 💡 时序的核心挑战：如何把 Transformer 的优势用对。

## 2. PatchTST

### 2.1 核心思想

```
PatchTST (Patch Time Series Transformer):
  1. 把时间序列分成 patch (块)
  2. 每个 patch 作为一个 token
  3. 用 Transformer 处理

为什么 patch?
  - 降维: L 个点 → L/P 个 patch
  - 语义: 一个 patch 包含局部模式
  - 效率: 序列更短 → 注意力更快

类比:
  ViT 把图像分 patch → PatchTST 把时序分 patch
```

### 2.2 架构

```
输入: x[1:L] (长度 L 的时序)
  ↓
分 patch: [x[1:P], x[P+1:2P], ...] (L/P 个 patch)
  ↓
线性嵌入: 每个 patch → d 维向量
  ↓
位置编码: + positional encoding
  ↓
Transformer: 多层自注意力
  ↓
线性头: → 预测未来 H 步
```

In [ ]:
# PatchTST 实现
class PatchTST(nn.Module):
    def __init__(self, seq_len=96, patch_len=16, stride=8, d_model=64, n_heads=4, n_layers=2, pred_len=24):
        super().__init__()
        self.patch_len = patch_len
        self.stride = stride
        self.pred_len = pred_len

        # 计算 patch 数量
        self.n_patches = (seq_len - patch_len) // stride + 1

        # Patch 嵌入
        self.patch_embed = nn.Linear(patch_len, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, self.n_patches, d_model) * 0.02)

        # Transformer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4,
            dropout=0.1, batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, n_layers)

        # 预测头
        self.head = nn.Linear(d_model * self.n_patches, pred_len)

    def forward(self, x):
        # x: (B, L)
        B = x.size(0)

        # 分 patch
        patches = []
        for i in range(self.n_patches):
            start = i * self.stride
            patch = x[:, start:start + self.patch_len]
            patches.append(patch)
        patches = torch.stack(patches, dim=1)  # (B, n_patches, patch_len)

        # 嵌入
        tokens = self.patch_embed(patches)  # (B, n_patches, d_model)
        tokens = tokens + self.pos_embed

        # Transformer
        out = self.transformer(tokens)  # (B, n_patches, d_model)

        # 预测
        out = out.reshape(B, -1)  # (B, n_patches * d_model)
        pred = self.head(out)  # (B, pred_len)
        return pred

# 生成时序数据
np.random.seed(42)
n = 1000
t = np.arange(n)
data = np.sin(t * 0.1) + 0.5 * np.sin(t * 0.05) + np.random.randn(n) * 0.1

# 准备数据
seq_len, pred_len = 96, 24
X, Y = [], []
for i in range(n - seq_len - pred_len):
    X.append(data[i:i+seq_len])
    Y.append(data[i+seq_len:i+seq_len+pred_len])
X = torch.tensor(np.array(X), dtype=torch.float32)
Y = torch.tensor(np.array(Y), dtype=torch.float32)

# 训练
model = PatchTST(seq_len=seq_len, patch_len=16, stride=8, pred_len=pred_len)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(50):
    pred = model(X)
    loss = F.mse_loss(pred, Y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(f"PatchTST 训练完成, 最终 MSE: {loss.item():.6f}")

# 预测可视化
with torch.no_grad():
    pred = model(X[:1])[0].numpy()
    actual = Y[0].numpy()
    context = X[0].numpy()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(seq_len), context, 'b-', label='历史')
ax.plot(range(seq_len, seq_len + pred_len), actual, 'g-', label='真实未来')
ax.plot(range(seq_len, seq_len + pred_len), pred, 'r--', label='PatchTST 预测')
ax.axvline(x=seq_len, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('时间步', fontsize=12)
ax.set_ylabel('值', fontsize=12)
ax.set_title('PatchTST 时序预测', fontsize=14, fontweight='bold')
ax.legend(fontsize=11); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('notebooks/fig_patchtst.png', bbox_inches='tight')
plt.show()

## 3. iTransformer

### 3.1 核心思想

```
iTransformer (inverted Transformer):
  反转维度: 把"时间"和"变量"角色互换

传统 Transformer:
  - 每个时间步是一个 token
  - 注意力在时间步之间
  - 多变量 → 在特征维拼接

iTransformer:
  - 每个变量是一个 token
  - 注意力在变量之间
  - 时间 → 在特征维拼接

直觉:
  多变量时序: x[t, v] (t=时间, v=变量)
  传统: 固定 v, 在 t 上做注意力
  iTransformer: 固定 t, 在 v 上做注意力
```

### 3.2 为什么有效？

```
传统方法的问题:
  - 时间步太多 → 注意力矩阵大
  - 变量间关系没被显式建模

iTransformer 优势:
  - 变量数 < 时间步 → 注意力更高效
  - 显式建模变量间关系
  - 每个变量的时间序列作为 embedding
```

> 💡 iTransformer 的洞察：在多变量时序中，变量间的关系比时间步间的关系更重要。

In [ ]:
# iTransformer 实现
class iTransformer(nn.Module):
    def __init__(self, n_vars=5, seq_len=96, d_model=64, n_heads=4, n_layers=2, pred_len=24):
        super().__init__()
        self.n_vars = n_vars
        self.pred_len = pred_len

        # 每个变量的时间序列 → embedding
        self.embed = nn.Linear(seq_len, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, n_vars, d_model) * 0.02)

        # Transformer (在变量维度做注意力)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model*4,
            dropout=0.1, batch_first=True, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, n_layers)

        # 预测头
        self.head = nn.Linear(d_model, pred_len)

    def forward(self, x):
        # x: (B, L, V) → 转置为 (B, V, L)
        x = x.transpose(1, 2)  # (B, V, L)

        # 每个变量的时间序列 → embedding
        tokens = self.embed(x)  # (B, V, d_model)
        tokens = tokens + self.pos_embed

        # Transformer (在变量维度)
        out = self.transformer(tokens)  # (B, V, d_model)

        # 预测
        pred = self.head(out)  # (B, V, pred_len)
        pred = pred.transpose(1, 2)  # (B, pred_len, V)
        return pred

# 多变量时序数据
np.random.seed(42)
n_vars = 5
data_multi = np.zeros((n, n_vars))
for v in range(n_vars):
    data_multi[:, v] = np.sin(t * 0.1 * (v+1)) + np.random.randn(n) * 0.1

X_multi, Y_multi = [], []
for i in range(n - seq_len - pred_len):
    X_multi.append(data_multi[i:i+seq_len])
    Y_multi.append(data_multi[i+seq_len:i+seq_len+pred_len])
X_multi = torch.tensor(np.array(X_multi), dtype=torch.float32)
Y_multi = torch.tensor(np.array(Y_multi), dtype=torch.float32)

# 训练 iTransformer
model_i = iTransformer(n_vars=n_vars, seq_len=seq_len, pred_len=pred_len)
optimizer_i = torch.optim.Adam(model_i.parameters(), lr=1e-3)

for epoch in range(50):
    pred = model_i(X_multi)
    loss = F.mse_loss(pred, Y_multi)
    optimizer_i.zero_grad()
    loss.backward()
    optimizer_i.step()

print(f"iTransformer 训练完成, 最终 MSE: {loss.item():.6f}")

# 可视化
with torch.no_grad():
    pred = model_i(X_multi[:1])[0].numpy()
    actual = Y_multi[0].numpy()

fig, ax = plt.subplots(figsize=(12, 5))
for v in range(min(3, n_vars)):
    ax.plot(range(pred_len), actual[:, v], '-', alpha=0.7, label=f'真实 变量{v+1}')
    ax.plot(range(pred_len), pred[:, v], '--', alpha=0.7, label=f'预测 变量{v+1}')
ax.set_xlabel('预测步', fontsize=12)
ax.set_ylabel('值', fontsize=12)
ax.set_title('iTransformer 多变量预测', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, ncol=3); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('notebooks/fig_itransformer.png', bbox_inches='tight')
plt.show()

## 4. 方法对比

### 4.1 总结

```
方法          优点              缺点
ARIMA         可解释            限平稳、单变量
LSTM          长序列            串行慢
Transformer   并行、长距离      直接用效果一般
PatchTST      局部模式、高效    patch 大小需调
iTransformer  变量关系、高效    适合多变量
```

In [ ]:
# 方法对比
fig, ax = plt.subplots(figsize=(10, 6))

methods = ['ARIMA', 'LSTM', 'Transformer', 'PatchTST', 'iTransformer']
accuracy = [0.6, 0.7, 0.72, 0.85, 0.87]
speed = [0.8, 0.3, 0.7, 0.8, 0.9]
multivariate = [0.2, 0.7, 0.7, 0.8, 0.9]

x = np.arange(len(methods))
width = 0.25

ax.bar(x - width, accuracy, width, label='准确率', color='steelblue', alpha=0.8)
ax.bar(x, speed, width, label='速度', color='coral', alpha=0.8)
ax.bar(x + width, multivariate, width, label='多变量支持', color='forestgreen', alpha=0.8)

ax.set_xlabel('方法', fontsize=12)
ax.set_ylabel('评分 (0-1)', fontsize=12)
ax.set_title('时序方法对比', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(methods, fontsize=10, rotation=15)
ax.legend(fontsize=11); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('notebooks/fig_timeseries_comparison.png', bbox_inches='tight')
plt.show()
print("PatchTST 和 iTransformer 在准确率和速度上都优于传统方法。")

## 5. 小结

### ✅ 本章你掌握了
| 概念 | 状态 |
|------|------|
| 时序任务类型 | ✅ |
| PatchTST (patch + Transformer) | ✅ |
| iTransformer (反转维度) | ✅ |
| vs 传统方法 | ✅ |

### 核心 takeaway
> **PatchTST 用 patch 捕捉局部模式，iTransformer 反转维度建模变量关系**——两者都是把 Transformer 用对地方的关键创新。

### 🔗 下一章
**`62_time_series_foundation.ipynb`** — TimesFM/Chronos/Moirai 时序基础模型

---

> 💬 **板块十(时序深度学习)进行中 (1/2)。**